# SFT-пайплайн: MiniGrid → dataset.json → nanoVLM fine-tuning

Полный цикл:
1. **Генерация** `Datasets/dataset.json` (процедурные лабиринты, curriculum)
2. **Эксперт BFS** — кратчайшие траектории `(image, mission) → action`
3. **Fine-tuning** предобученного `nanoVLM-222M` (все веса, два LR)
4. **Метрики** в консоль + графики loss / accuracy
5. **Сохранение** чекпоинта и пробный inference

> Запускайте ячейки сверху вниз. Kernel: `.venv` проекта.

In [ ]:
!git clone https://github.com/berkutivan/NanoVLM
%cd /content/NanoVLM
!unzip /content/drive/MyDrive/Tbank/checkpoints.zip
!pip install -r requirements.txt -quiet
%cd /content/NanoVLM/Training_model

## 0. Пути и зависимости

In [ ]:
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Корень репозитория Tbank
ROOT = Path.cwd().resolve()
if ROOT.name == "Training_model":
    ROOT = ROOT.parent

DATASETS_DIR = ROOT / "Datasets"
NANOVLM_DIR = ROOT / "nanoVLM"
TRAINING_DIR = ROOT / "Training_model"
DATASET_JSON = DATASETS_DIR / "dataset.json"
PRETRAINED_CKPT = ROOT / "checkpoints" / "nanoVLM-222M"
SFT_OUTPUT_DIR = ROOT / "checkpoints" / "sft-minigrid"

for p in (str(NANOVLM_DIR), str(DATASETS_DIR), str(TRAINING_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print("ROOT:", ROOT)
print("CUDA:", torch.cuda.is_available(), "| device:", "cuda" if torch.cuda.is_available() else "cpu")

## 1. Генерация `dataset.json`

Каждый объект = одно процедурное поле (`ProceduralMazeEnv`):
- `layout` — стены, цель, старт агента
- `mission` — текст задачи
- `predicate_space` — действия, размер сетки, формат RGB-наблюдения

Curriculum: сначала маленькие карты без стен, затем сложнее.

In [ ]:
from maze_presample import (
    CURRICULUM,
    MazePreSampleBuilder,
    MazePreSampleConfig,
    load_dataset,
    save_dataset,
    curriculum_for_index,
)


def generate_dataset(n_mazes: int, overwrite: bool = False) -> dict:
    """Создать или дополнить dataset.json с n_mazes лабиринтами."""
    if overwrite or not DATASET_JSON.exists():
        data = {"version": 2, "format": "pre_sample", "objects": []}
    else:
        data = load_dataset()

    start_id = len(data["objects"])
    target = start_id + n_mazes

    while len(data["objects"]) < target:
        obj_id = len(data["objects"])
        cur = curriculum_for_index(obj_id)
        cfg = MazePreSampleConfig(
            size=cur["size"],
            n_walls=cur["n_walls"],
            seed=obj_id,
        )
        sample = MazePreSampleBuilder(cfg).build()
        sample["id"] = obj_id
        data["objects"].append(sample)
        print(
            f"  id={obj_id} size={cur['size']} walls={cur['n_walls']} "
            f"goal={sample['layout']['goal_pos']}"
        )

    save_dataset(data)
    print(f"Saved {len(data['objects'])} objects -> {DATASET_JSON}")
    return data


# --- настройка ---
N_MAZES = 100          # сколько лабиринтов (100 уже есть? поставьте 0 и skip)
OVERWRITE = False      # True = пересоздать json с нуля

if N_MAZES > 0 or OVERWRITE:
    print("Generating dataset...")
    dataset = generate_dataset(N_MAZES, overwrite=OVERWRITE)
else:
    dataset = load_dataset()
    print(f"Using existing dataset: {len(dataset['objects'])} objects")

In [ ]:
# Быстрый просмотр одного объекта
obj = dataset["objects"][0]
print(json.dumps({k: obj[k] for k in ("id", "mission", "layout", "predicate_space")}, indent=2))

## 2. Визуализация поля и экспертная траектория (BFS)

Эксперт: **BFS** по состоянию `(x, y, direction)` — кратчайший путь в лабиринте.
Кадры RGB снимаются через `RGBImgPartialObsWrapper` **на лету** (не хранятся в json).

In [ ]:
from maze_expert import plan_expert_actions, rollout_expert_trajectory

demo_obj = dataset["objects"][0]
actions = plan_expert_actions(demo_obj)
trajectory = rollout_expert_trajectory(demo_obj)

print(f"Expert path length: {len(actions)} steps")
print("Actions:", " -> ".join(actions))

fig, axes = plt.subplots(1, min(5, len(trajectory)), figsize=(3 * min(5, len(trajectory)), 3))
if len(trajectory) == 1:
    axes = [axes]
for ax, step in zip(axes, trajectory[:5]):
    ax.imshow(step.image)
    ax.set_title(f"step {step.step_index}: {step.action}")
    ax.axis("off")
plt.suptitle(demo_obj["mission"])
plt.tight_layout()
plt.show()

## 3. Загрузка предобученного nanoVLM-222M

Веса должны лежать в `checkpoints/nanoVLM-222M/` (`config.json` + `model.safetensors`).
Если папки нет — скачаем с Hugging Face Hub.

In [ ]:
from models.vision_language_model import VisionLanguageModel
from data.processors import get_image_processor, get_tokenizer

HUB_ID = "lusxvr/nanoVLM-222M"
source = str(PRETRAINED_CKPT) if PRETRAINED_CKPT.exists() else HUB_ID

print(f"Loading VLM from: {source}")
model = VisionLanguageModel.from_pretrained(source)
n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

tokenizer = get_tokenizer(model.cfg.lm_tokenizer)
image_processor = get_image_processor(model.cfg.vit_img_size)
print("Tokenizer:", model.cfg.lm_tokenizer)
print("Max text length:", model.cfg.lm_max_length)

## 4. SFT-датасет и collator

Каждый **шаг траектории** = один training sample:
- **Вход:** RGB-кадр + промпт с mission
- **Цель:** одно слово `left` / `right` / `forward`
- **Loss:** cross-entropy только на токенах ответа (промпт замаскирован `-100`)

In [ ]:
from minigrid_sft_dataset import (
    MiniGridSFTDataset,
    load_objects,
    precompute_trajectories,
    split_objects,
    PROMPT_TEMPLATE,
)
from minigrid_collator import MiniGridSFTCollator

objects = load_objects(DATASET_JSON)
MAX_OBJECTS = None   # None = все; для быстрого теста поставьте 20
if MAX_OBJECTS is not None:
    objects = objects[:MAX_OBJECTS]

VAL_RATIO = 0.1
SEED = 0
train_objs, val_objs = split_objects(objects, VAL_RATIO, SEED)
print(f"Mazes: total={len(objects)} train={len(train_objs)} val={len(val_objs)}")

print("Precomputing BFS trajectories...")
t0 = time.time()
train_cache = precompute_trajectories(train_objs)
val_cache = precompute_trajectories(val_objs)
train_steps = sum(len(v) for v in train_cache.values())
val_steps = sum(len(v) for v in val_cache.values())
print(f"SFT samples: train={train_steps}, val={val_steps} ({time.time()-t0:.1f}s)")

train_ds = MiniGridSFTDataset(train_objs, tokenizer, image_processor, train_cache)
val_ds = MiniGridSFTDataset(val_objs, tokenizer, image_processor, val_cache)
collator = MiniGridSFTCollator(tokenizer, model.cfg.lm_max_length)

sample = train_ds[0]
print("\nПример промпта:")
print(sample["text_data"])
print("Ответ:", sample["answer"].strip())

In [ ]:
plt.imshow(sample["image"].permute(1, 2, 0).numpy())
plt.title(f"object_id={sample['object_id']} step={sample['step_index']} action={sample['action']}")
plt.axis("off")
plt.show()

## 5. Стратегия fine-tuning (только верхние слои)

По умолчанию в nanoVLM можно обучать **все** параметры, но для стабильного и быстрого fine-tuning мы замораживаем почти всё.

| Что | Как |
|-----|-----|
| Инициализация | `VisionLanguageModel.from_pretrained(...)` — **полные** веса VLM (vision + LM + MP) |
| Обучаем | **Только верхние слои**: последние блоки ViT + последние блоки LM + `MP` + `lm_head`/нормы |
| Замораживаем | Остальные слои vision+LM (и эмбеддинги) |
| LR группы | MP (`lr_mp`) выше, “верхние backbone-слои” (`lr_backbones`) ниже |
| Schedule | Cosine decay + 3% warmup |
| Loss | Causal LM CE, маска на промпте (`labels=-100`) |
| AMP | `bfloat16` на CUDA, иначе float32 |

In [ ]:
# --- гиперпараметры ---
EPOCHS = 3
BATCH_SIZE = 8 if torch.cuda.is_available() else 2
GRAD_ACCUM = 1

# Fine-tune только верхние слои
UNFREEZE_VIT_BLOCKS = 2   # сколько последних ViT блоков обучать
UNFREEZE_LM_BLOCKS = 2    # сколько последних LM блоков обучать

LR_MP = 1e-3
LR_BACKBONES = 5e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
LOG_EVERY = 10
EVAL_EVERY = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
amp_dtype = torch.bfloat16


def set_requires_grad(module, value: bool) -> None:
    for p in module.parameters():
        p.requires_grad = value


def freeze_for_sft(model: VisionLanguageModel, *, vit_last_n: int, lm_last_n: int) -> None:
    # 1) Freeze everything
    set_requires_grad(model, False)

    # 2) Always train modality projector
    set_requires_grad(model.MP, True)

    # 3) Unfreeze last N ViT blocks + final layer norm
    vit = model.vision_encoder
    if hasattr(vit, "blocks"):
        n = len(vit.blocks)
        for i in range(max(0, n - vit_last_n), n):
            set_requires_grad(vit.blocks[i], True)
    if hasattr(vit, "layer_norm"):
        set_requires_grad(vit.layer_norm, True)

    # 4) Unfreeze last N LM blocks + final norm + head
    dec = model.decoder
    if hasattr(dec, "blocks"):
        n = len(dec.blocks)
        for i in range(max(0, n - lm_last_n), n):
            set_requires_grad(dec.blocks[i], True)
    if hasattr(dec, "norm"):
        set_requires_grad(dec.norm, True)
    if hasattr(dec, "head"):
        set_requires_grad(dec.head, True)


freeze_for_sft(model, vit_last_n=UNFREEZE_VIT_BLOCKS, lm_last_n=UNFREEZE_LM_BLOCKS)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_trainable:,} / {sum(p.numel() for p in model.parameters()):,}")

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
    pin_memory=device.type == "cuda",
    drop_last=len(train_ds) >= BATCH_SIZE,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

# Optimizer: берём только параметры с requires_grad=True
mp_params = [p for p in model.MP.parameters() if p.requires_grad]
backbone_params = [p for p in list(model.decoder.parameters()) + list(model.vision_encoder.parameters()) if p.requires_grad]

param_groups = [
    {"params": mp_params, "lr": LR_MP},
    {"params": backbone_params, "lr": LR_BACKBONES},
]
optimizer = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
model.to(device)

max_steps = max(1, (len(train_loader) // GRAD_ACCUM) * EPOCHS)
print(f"Device: {device} | batches/epoch: {len(train_loader)} | max_steps: {max_steps}")

## 6. Обучение SFT + метрики

In [ ]:
def get_lr(step: int, max_lr: float, max_steps: int) -> float:
    min_lr = max_lr * 0.1
    warmup = max(1, int(max_steps * 0.03))
    if step < warmup:
        return max_lr * (step + 1) / warmup
    if step >= max_steps:
        return min_lr
    decay = (step - warmup) / max(max_steps - warmup, 1)
    return min_lr + 0.5 * (1 + math.cos(math.pi * decay)) * (max_lr - min_lr)


def normalize_action(text: str) -> str:
    t = text.strip().lower()
    for name in ("forward", "left", "right"):
        if name in t:
            return name
    return t.split()[0] if t else ""


def _make_action_sequences(tokenizer, prompts: list[str], actions: list[str]):
    # Note: keep exact formatting consistent with dataset: leading space + eos
    answers = [f" {a}{tokenizer.eos_token}" for a in actions]
    return [prompts[i] + answers[i] for i in range(len(prompts))]


@torch.no_grad()
def choose_action_from_allowed(model, tokenizer, images, prompts: list[str], allowed: list[list[str]]):
    """Pick the most likely action among allowed actions (no free-form generation)."""
    # Flatten options
    flat_prompts: list[str] = []
    flat_actions: list[str] = []
    owner: list[int] = []
    for i, acts in enumerate(allowed):
        acts2 = acts if acts else ["left", "right", "forward"]
        for a in acts2:
            owner.append(i)
            flat_prompts.append(prompts[i])
            flat_actions.append(a)

    seqs = _make_action_sequences(tokenizer, flat_prompts, flat_actions)
    enc = tokenizer(
        seqs,
        padding=True,
        padding_side="left",
        return_tensors="pt",
        truncation=True,
        max_length=model.cfg.lm_max_length,
    )
    input_ids = enc["input_ids"].to(images.device)
    attention_mask = enc["attention_mask"].to(images.device)

    # Build labels: predict next token, mask the prompt
    labels = input_ids.clone()
    labels[:, :-1] = input_ids[:, 1:].clone()
    labels[:, -1] = -100

    # Mask prompt portion
    prompt_lens = [len(tokenizer.encode(p, add_special_tokens=False)) for p in flat_prompts]
    for j in range(len(flat_prompts)):
        first_token_pos = attention_mask[j].nonzero(as_tuple=True)[0][0].item()
        q_end = first_token_pos + prompt_lens[j] - 1
        labels[j, :q_end] = -100

    # Repeat images per option
    rep_images = images[owner]

    # Compute per-option NLL via logits (sum over answer tokens)
    embd, _ = model(input_ids, rep_images, attention_mask=attention_mask, targets=None)
    logits = model.decoder.head(embd)

    # Only token positions corresponding to text (exclude image token prefix)
    img_len = model.MP(model.vision_encoder(rep_images[:1])).size(1)
    logits_text = logits[:, img_len:, :]  # [B*K, seq_len, vocab]

    nlls: list[float] = []
    for j in range(labels.size(0)):
        mask = labels[j] != -100
        if mask.sum().item() == 0:
            nlls.append(1e9)
            continue
        tgt = labels[j][mask]
        pred = logits_text[j][mask]
        nll = torch.nn.functional.cross_entropy(pred, tgt, reduction="sum")
        nlls.append(float(nll.item()))

    # Select best action per original sample
    best = [None] * len(prompts)
    best_nll = [1e30] * len(prompts)
    for j, i in enumerate(owner):
        if nlls[j] < best_nll[i]:
            best_nll[i] = nlls[j]
            best[i] = flat_actions[j]

    return best


@torch.no_grad()
def eval_metrics(model, tokenizer, val_loader, val_ds, device, max_new_tokens=8):
    model.eval()
    total_loss, n_batches = 0.0, 0
    for batch in val_loader:
        images = batch["image"].to(device)
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        _, loss = model(input_ids, images, attention_mask=attention_mask, targets=labels)
        total_loss += loss.item()
        n_batches += 1

    # Metric: model must output one of allowed actions
    in_allowed, total = 0, 0
    bs = val_loader.batch_size or BATCH_SIZE
    for start in range(0, len(val_ds), bs):
        items = [val_ds[i] for i in range(start, min(start + bs, len(val_ds)))]
        images = torch.stack([it["image"] for it in items]).to(device)
        prompts = [it["text_data"] for it in items]
        allowed = [it.get("allowed_actions") or [it["action"]] for it in items]

        picked = choose_action_from_allowed(model, tokenizer, images, prompts, allowed)
        for pred, acts in zip(picked, allowed):
            pred_n = normalize_action(pred or "")
            in_allowed += int(pred_n in set(acts))
            total += 1

    model.train()
    return total_loss / max(n_batches, 1), in_allowed / max(total, 1)


# история метрик
history = {"step": [], "train_loss": [], "val_loss": [], "val_acc": []}
global_step = 0
best_val_acc = -1.0
SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("SFT fine-tuning started")
print("=" * 60)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss, epoch_batches = 0.0, 0
    t_epoch = time.time()

    for batch_idx, batch in enumerate(train_loader):
        images = batch["image"].to(device)
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                _, loss = model(input_ids, images, attention_mask=attention_mask, targets=labels)
        else:
            _, loss = model(input_ids, images, attention_mask=attention_mask, targets=labels)

        (loss / GRAD_ACCUM).backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.param_groups[0]["lr"] = get_lr(global_step, LR_MP, max_steps)
            optimizer.param_groups[1]["lr"] = get_lr(global_step, LR_BACKBONES, max_steps)
            optimizer.step()
            global_step += 1

        loss_val = loss.item()
        epoch_loss += loss_val
        epoch_batches += 1

        if global_step > 0 and global_step % LOG_EVERY == 0:
            print(
                f"[ep {epoch+1}/{EPOCHS}] step {global_step} | "
                f"loss={loss_val:.4f} | lr_mp={optimizer.param_groups[0]['lr']:.2e} "
                f"lr_bb={optimizer.param_groups[1]['lr']:.2e}"
            )
            history["step"].append(global_step)
            history["train_loss"].append(loss_val)

        if global_step > 0 and global_step % EVAL_EVERY == 0:
            val_loss, val_acc = eval_metrics(model, tokenizer, val_loader, val_ds, device)
            print(f"  >> val_loss={val_loss:.4f} | val_allowed_acc={val_acc*100:.2f}%")
            history["val_loss"].append(val_loss)
            history["val_acc"].append(val_acc)
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                model.save_pretrained(str(SFT_OUTPUT_DIR / "best"))
                print(f"  >> saved best -> {SFT_OUTPUT_DIR / 'best'}")

    val_loss, val_acc = eval_metrics(model, tokenizer, val_loader, val_ds, device)
    print(
        f"Epoch {epoch+1} ({time.time()-t_epoch:.0f}s) | "
        f"avg_train_loss={epoch_loss/max(epoch_batches,1):.4f} | "
        f"val_loss={val_loss:.4f} | val_acc={val_acc*100:.2f}%"
    )
    model.save_pretrained(str(SFT_OUTPUT_DIR / f"epoch_{epoch+1}"))

model.save_pretrained(str(SFT_OUTPUT_DIR / "last"))
print("=" * 60)
print(f"Done. Best val acc: {best_val_acc*100:.2f}%")
print(f"Checkpoints: {SFT_OUTPUT_DIR}")
print("=" * 60)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if history["step"]:
    axes[0].plot(history["step"], history["train_loss"], label="train loss")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("loss")
    axes[0].set_title("Training loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

eval_steps = history["step"][-len(history["val_acc"]):] if history["val_acc"] else []
if history["val_acc"]:
    axes[1].plot(eval_steps, [a * 100 for a in history["val_acc"]], "o-", color="green", label="val action acc %")
    if history["val_loss"]:
        ax2 = axes[1].twinx()
        ax2.plot(eval_steps, history["val_loss"], "s--", color="red", alpha=0.7, label="val loss")
        ax2.set_ylabel("val loss", color="red")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("accuracy %")
    axes[1].set_title("Validation metrics")
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Inference: проверка обученной модели

In [ ]:
ckpt = SFT_OUTPUT_DIR / "best"
if not ckpt.exists():
    ckpt = SFT_OUTPUT_DIR / "last"

print(f"Loading fine-tuned weights: {ckpt}")
ft_model = VisionLanguageModel.from_pretrained(str(ckpt)).to(device)
ft_model.eval()

test_item = val_ds[0]
image = test_item["image"].unsqueeze(0).to(device)
prompt = test_item["text_data"]
gold = test_item["action"]

enc = tokenizer([prompt], padding=True, return_tensors="pt", truncation=True,
                max_length=ft_model.cfg.lm_max_length)
input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)

with torch.no_grad():
    gen = ft_model.generate(input_ids, image, attention_mask, max_new_tokens=8)
pred = tokenizer.batch_decode(gen, skip_special_tokens=True)[0]

print(f"Gold: {gold}")
print(f"Pred: {pred}")
print(f"Match: {normalize_action(pred) == gold}")

plt.imshow(test_item["image"].permute(1, 2, 0).numpy())
plt.title(f"pred={normalize_action(pred)} | gold={gold}")
plt.axis("off")
plt.show()

## Следующий шаг

Чекпоинт `checkpoints/sft-minigrid/best` используйте как инициализацию для **GRPO**-дообучения в среде MiniGrid.